# Causal self-attention: predicting the next token at every position

This notebook builds causal self-attention that predicts the **next token at
every position of a sequence simultaneously** — not one query at the end
(that was example 03), but the actual autoregressive loop a language model
runs: given `tokens[0..i]`, predict `token[i+1]`, for every `i` at once.

It is the notebook companion to
[`../examples/04_causal_series_predictor.py`](../examples/04_causal_series_predictor.py),
a verified, gradient-checked (to ~1.4e-6), self-contained NumPy script — no
autograd, no ML framework. This notebook is the same real math, the same
code, broken into cells for readers who'd rather step through it than read
one long file. It builds on example 03's five-step attention derivation
(score → separate Q from K → softmax weights → V-weighted retrieval →
1/√d_k scaling, part-1-maths.html §11) by adding §11's **causal masking**
subsection on top: a position may only look at itself and earlier
positions, never the future.

**The task**: a length-8 alternating two-symbol series, e.g.
`[a,b,a,b,a,b,a,b]`. Simplest possible sequence that makes causal masking
*necessary* — predicting the next symbol requires having seen both `a` and
`b` at least once, which is exactly what the mask controls access to.

### Why position 0 is unguessable

Predicting `token[1]` from `token[0]` alone means predicting `b` having
only seen `a`. `b` is an independent random draw (any of the other
`vocab_size - 1` symbols, uniformly) — nothing in `token[0]` determines it.
This is the causal mask working exactly as intended: position 0 genuinely
does not have the information yet. From position 1 onward the model has
seen both `a` and `b`, so the pattern is fully determined — the correct
move is "copy the token from one position back" (`token[i+1] == token[i-1]`
always, since the series has period 2) — and accuracy hits ~100% there.

### Confidently wrong, not randomly wrong

A subtlety actually observed when this was trained (not merely predicted in
advance): position 0 does not converge to a *chance*-level guess (~20% for
a 6-symbol vocabulary). The single linear `Wv`/`Wout` pair is shared across
every position, and to solve positions 1–6 it has to learn a
content-identity map — "whatever token's embedding you just retrieved,
output that token's class" — because the retrieved token literally *is*
the answer there. Position 0 is forced (by the causal mask — it is the
only valid key) to attend to itself, so it retrieves its *own* embedding,
and that same identity map confidently predicts "next token == current
token", which is *always* wrong (`a != b` by construction). So position 0
lands near **0% accuracy, not ~20% chance** — confidently wrong rather than
randomly wrong. Still not a bug: it's the direct, provable consequence of
the shared linear readout doing exactly what it needs to do everywhere
else.

### A correction worth flagging

An earlier draft of this example's description (and a companion browser
widget) assumed a trained model would attend to a single fixed position
("N positions back"). The actual, verified behavior is different: a
period-2 series has *multiple* earlier positions carrying the same correct
value, and a well-trained model spreads attention across **all** of them
(content-based matching, the same way example 03's retrieval worked) —
not onto one fixed offset. Section 7 below reproduces the exact
sample-prediction output that shows this.

In [1]:
import numpy as np

VOCAB_SIZE = 6
SEQ_LEN = 8
D_MODEL = 8

## The task: alternating a, b, a, b, ...

Pick two distinct symbols `a != b` from the vocabulary and a random
starting phase, then alternate: `[a,b,a,b,...]` or `[b,a,b,a,...]` for
`seq_len` tokens. It's the simplest series where "predict the next token"
is well-defined *and* requires history — a bare unigram model can't do
better than chance, and nothing before position 1 can possibly reveal `b`.
That's exactly the gap causal masking exists to enforce, not paper over.

In [2]:
def make_sequence(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, seed=None, rng=None):
    """One alternating a,b,a,b,... sequence. Pass `rng` (a np.random.Generator)
    to draw from a shared, advancing stream when generating many sequences
    (e.g. one per training example in a batch); pass `seed` for a single
    reproducible one-off sequence. Returns an (seq_len,) int64 array."""
    if rng is None:
        rng = np.random.default_rng(seed)
    a, b = rng.choice(vocab_size, size=2, replace=False)  # distinct by construction
    first, second = (a, b) if rng.integers(0, 2) == 0 else (b, a)  # random phase
    seq = np.empty(seq_len, dtype=np.int64)
    seq[0::2] = first
    seq[1::2] = second
    return seq


def make_batch(batch_size, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, rng=None):
    """A fresh, independent alternating sequence per row."""
    if rng is None:
        rng = np.random.default_rng()
    return np.stack([make_sequence(vocab_size, seq_len, rng=rng) for _ in range(batch_size)])


# sanity check
demo_seq = make_sequence(seed=42)
print(demo_seq)

[0 4 0 4 0 4 0 4]


## Parameters

Single head, `d_k = d_model` (no subspace split — keep the mechanics
visible). Token embedding `E_tok` and positional embedding `E_pos` are
summed, same as example 03; `Wq`, `Wk`, `Wv` are the three projections;
`Wout`/`bout` read the retrieved context back out to vocabulary logits.

In [3]:
def init_params(seed=0, d_model=D_MODEL, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN):
    """Small random init. d_k = d_model (single head, no subspace split)."""
    rng = np.random.default_rng(seed)

    def small(*shape):
        return rng.normal(0, 1.0 / np.sqrt(shape[-1]), shape)

    params = {
        "E_tok": rng.normal(0, 0.1, (vocab_size, d_model)),   # token embedding
        "E_pos": rng.normal(0, 0.1, (seq_len, d_model)),      # positional embedding
        "Wq": small(d_model, d_model),
        "Wk": small(d_model, d_model),
        "Wv": small(d_model, d_model),
        "Wout": small(vocab_size, d_model),   # (out, in), matches part-2's convention
        "bout": np.zeros(vocab_size),
        "meta": {"d_model": d_model, "vocab_size": vocab_size, "seq_len": seq_len},
    }
    return params

## Forward-pass building blocks

`softmax` uses the standard log-sum-exp shift for numerical stability.
`cross_entropy_loss` averages over *both* the batch and every predicted
position (there are `seq_len - 1` of those per sequence — the last
position has no "next token" to check against). `_causal_mask` builds the
additive mask that does the actual causal restriction: `0` where the key
position `j <= i` (allowed), `-1e9` where `j > i` (future). Added to the
raw attention scores before softmax, `e^-1e9` underflows to *exactly*
`0.0` — masked positions get literal zero weight, not just "very small".

In [4]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)   # log-sum-exp trick: no overflow, same result
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def cross_entropy_loss(probs, targets):
    """probs: (B,T,vocab) predicted-position probabilities; targets: (B,T)."""
    B, T = targets.shape
    ib = np.arange(B)[:, None]
    it = np.arange(T)[None, :]
    return -np.mean(np.log(probs[ib, it, targets] + 1e-12))


def _causal_mask(L):
    """(L,L) additive mask: 0 where j<=i (allowed), -1e9 where j>i (future).
    e^-1e9 underflows to exactly 0.0 in the softmax below -- the masked
    positions get literal zero weight, not just "very small"."""
    return np.triu(np.full((L, L), -1e9), k=1)

## Forward: §11's five steps, at every position at once

Concretely, *why* can position `i` only see keys `j <= i`? Because at
inference time, position `i`'s job is to predict `token[i+1]` using only
`tokens[0..i]` — the future genuinely isn't available yet, so training must
mirror that or the model would learn to cheat (attend to the very token
it's supposed to predict). The mask enforces it by adding `-1e9` to every
score `(i, j)` with `j > i` *before* the softmax, so those keys get exactly
`0` weight — a "leak-proof" restriction rather than a soft discouragement.

Everything else is example 03's Q/K/V mechanics, just batched over every
query position `i` (and every batch row) simultaneously instead of one:
`scores[b,i,j] = Q_i . K_j / sqrt(d_k)`, masked, softmaxed per row `i` over
its own valid `j` range, then used to weight-sum `V` into each position's
retrieved `context`.

In [5]:
def forward(tokens, params):
    """tokens: (B,L) or (L,) int array. Returns (logits, attn_weights, cache).
    logits: (B,L,vocab) -- next-token prediction at every position (the
    prediction FROM the last position has no target to check against, but
    is still computed/returned).
    attn_weights: (B,L,L) causally-masked attention, weights[b,i,j] = how
    much query position i attends to key position j (0 for all j>i)."""
    if tokens.ndim == 1:
        tokens = tokens[None, :]
    B, L = tokens.shape
    E_tok, E_pos = params["E_tok"], params["E_pos"]
    Wq, Wk, Wv, Wout, bout = params["Wq"], params["Wk"], params["Wv"], params["Wout"], params["bout"]
    d_k = Wq.shape[1]

    X = E_tok[tokens] + E_pos[None, :L, :]              # (B,L,d) -- token + positional embedding

    # Step 2 -- separate "what I seek" (Q) from "what I am" (K); every
    # position is simultaneously a query AND a key/value provider now.
    Q = X @ Wq                                          # (B,L,d_k)
    K = X @ Wk                                          # (B,L,d_k)
    V = X @ Wv                                          # (B,L,d_v)

    # Step 1/5 -- score every (query i, key j) pair, scaled by 1/sqrt(d_k)
    scores = np.einsum("bid,bjd->bij", Q, K) / np.sqrt(d_k)   # (B,L,L)
    scores = scores + _causal_mask(L)[None, :, :]             # future positions -> -inf

    # Step 3 -- softmax each query row i over its own valid key range j<=i
    weights = softmax(scores, axis=-1)                  # (B,L,L), rows sum to 1, upper triangle exactly 0

    # Step 4 -- weighted sum of V: what each position actually retrieves
    context = np.einsum("bij,bjd->bid", weights, V)      # (B,L,d_v)

    logits = context @ Wout.T + bout                     # (B,L,vocab)
    probs = softmax(logits, axis=-1)

    cache = {"tokens": tokens, "X": X, "Q": Q, "K": K, "V": V,
              "scores": scores, "weights": weights, "context": context, "probs": probs}
    return logits, weights, cache

## Backward: every step undone by hand

Loss = mean cross-entropy over the 7 predicted positions (`0..L-2`, each
predicting the very next token) and over the batch — so the gradient at
the output is scaled by `1 / (B * n_pred)` instead of just `1 / B`,
because there are `n_pred` loss terms contributing per sequence, not one.

The softmax-over-valid-keys backward step is the *same identity* as
example 03's single-query case (`s_i * (delta_i - sum_k s_k * delta_k)`),
just applied **per query position `i`, over its own valid range of keys**
— rows are independent, and the masked (zero-weight) entries automatically
carry zero gradient since they multiply by `weights == 0`. No special-casing
needed; the mask that zeroed the forward pass zeroes the backward pass too.

From there it's the same §6–7 backprop identities as everywhere else in
this course: `delta = blame`, `dL/dW = delta @ x^T` (summed over batch and
position here), `dL/db = delta` (summed likewise), `dL/dx = W^T @ delta`
— applied three times over (once each for the Q, K, V projection paths),
since every position is simultaneously a query, a key, *and* a value here.

In [6]:
def backward(params, cache, tokens):
    Wq, Wk, Wv, Wout = params["Wq"], params["Wk"], params["Wv"], params["Wout"]
    X, weights, V, K, Q = cache["X"], cache["weights"], cache["V"], cache["K"], cache["Q"]
    context, probs = cache["context"], cache["probs"]
    B, L, d = X.shape
    d_k = Wq.shape[1]
    n_pred = L - 1                                       # positions 0..L-2 have a real next token

    # ---- output projection + softmax + cross-entropy (§6-7: collapses to
    # exactly predicted - actual) -- only the n_pred positions that have a
    # target contribute; the last position's logits get zero gradient. ----
    targets = tokens[:, 1:]                               # (B, n_pred) = token[i+1] for i=0..L-2
    delta_logits = np.zeros_like(probs)                    # (B,L,vocab)
    delta_valid = probs[:, :n_pred, :].copy()
    ib = np.arange(B)[:, None]
    it = np.arange(n_pred)[None, :]
    delta_valid[ib, it, targets] -= 1
    delta_valid /= (B * n_pred)                            # average over batch AND predicted positions
    delta_logits[:, :n_pred, :] = delta_valid

    dWout = np.einsum("bic,bid->cd", delta_logits, context)   # dL/dW = delta x^T, summed over batch & position
    dbout = delta_logits.sum(axis=(0, 1))                      # dL/db = delta, summed likewise
    dcontext = delta_logits @ Wout                               # dL/dx = W^T delta

    # ---- Step 4 backward: context[i] = sum_j weights[i,j] * V[j] ----
    dweights = np.einsum("bid,bjd->bij", dcontext, V)           # (B,L,L)
    dV = np.einsum("bij,bid->bjd", weights, dcontext)            # (B,L,d_v)

    # ---- Step 3 backward: softmax per query row i, over its key axis j.
    # Same identity as §5 (s_i(delta_i - sum_k s_k delta_k)); masked
    # positions have weight 0 so they automatically get zero gradient. ----
    dot = (weights * dweights).sum(axis=-1, keepdims=True)      # (B,L,1)
    dscores = weights * (dweights - dot)                         # (B,L,L)

    # ---- Step 1/5 backward: scores[i,j] = (Q_i . K_j) / sqrt(d_k) ----
    dQ = np.einsum("bij,bjd->bid", dscores, K) / np.sqrt(d_k)    # (B,L,d_k)
    dK = np.einsum("bij,bid->bjd", dscores, Q) / np.sqrt(d_k)    # (B,L,d_k)

    # ---- Step 2/4 backward: the three projections, back to dL/dW = delta x^T ----
    dWq = np.einsum("bld,ble->de", X, dQ)                         # (d, d_k), summed over batch & position
    dWk = np.einsum("bld,ble->de", X, dK)
    dWv = np.einsum("bld,ble->de", X, dV)

    # dL/dx = W^T delta, accumulated across all three projection paths --
    # every position is a query AND a key AND a value here, so all three
    # contribute to every position's embedding gradient.
    dX = dQ @ Wq.T + dK @ Wk.T + dV @ Wv.T                        # (B,L,d)

    # ---- embeddings: X = E_tok[tokens] + E_pos, blame scattered back to
    # whichever rows were actually looked up ----
    dE_tok = np.zeros_like(params["E_tok"])
    np.add.at(dE_tok, tokens.reshape(-1), dX.reshape(-1, d))
    dE_pos = dX.sum(axis=0)                                        # E_pos broadcasts over batch -> sum blame over batch

    return {"E_tok": dE_tok, "E_pos": dE_pos, "Wq": dWq, "Wk": dWk, "Wv": dWv,
            "Wout": dWout, "bout": dbout}

## Position-broken-out accuracy

Position 0 is reported **separately** from positions 1–6 throughout this
notebook — see "why position 0 is unguessable" above. A single blended
accuracy number (which would sit around `6/7 ≈ 86%` at best) would look
like a partial failure when the model is in fact working exactly as
designed.

In [7]:
def position_accuracy(tokens, params):
    """Returns (pos0_acc, pos1_6_acc, overall_loss) for a batch of sequences."""
    logits, _, cache = forward(tokens, params)
    n_pred = tokens.shape[1] - 1
    preds = logits[:, :n_pred, :].argmax(axis=-1)         # (B, n_pred)
    targets = tokens[:, 1:]
    correct = preds == targets                             # (B, n_pred)
    pos0_acc = correct[:, 0].mean()
    pos1_6_acc = correct[:, 1:].mean()
    loss = cross_entropy_loss(cache["probs"][:, :n_pred, :], targets)
    return pos0_acc, pos1_6_acc, loss

## Training and inference

Plain SGD, fresh random batch every step (no fixed dataset to overfit to
— the task is cheap enough to generate on the fly). `predict` runs
`forward` and returns the argmax prediction at every position plus the
attention weights, for inspection.

(The standalone script also persists trained weights to a `.npz` file
between runs; this notebook keeps everything in memory for a single linear
read-through, so `train` below returns `params` directly instead of
writing to disk.)

In [8]:
def train(steps=900, lr=0.5, batch_size=16, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN,
          d_model=D_MODEL, seed=0, verbose=True, print_every=100):
    params = init_params(seed=seed, d_model=d_model, vocab_size=vocab_size, seq_len=seq_len)
    rng = np.random.default_rng(seed + 1)              # drives training-batch generation
    eval_rng = np.random.default_rng(seed + 2)          # separate stream for periodic held-out reporting
    grad_keys = ["E_tok", "E_pos", "Wq", "Wk", "Wv", "Wout", "bout"]

    for step in range(1, steps + 1):
        tokens = make_batch(batch_size, vocab_size, seq_len, rng=rng)   # fresh random sequences every step
        logits, _, cache = forward(tokens, params)
        grads = backward(params, cache, tokens)
        for k in grad_keys:
            params[k] -= lr * grads[k]

        if verbose and (step % print_every == 0 or step == steps):
            eval_tokens = make_batch(500, vocab_size, seq_len, rng=eval_rng)
            pos0_acc, pos1_6_acc, loss = position_accuracy(eval_tokens, params)
            print(f"step {step:4d}   loss {loss:.4f}   pos0 acc {pos0_acc:.3f}   pos1-6 acc {pos1_6_acc:.3f}")

    return params


def predict(tokens, params):
    """tokens: (L,) single sequence or (B,L) batch of already-encoded token
    ids. Returns (predicted_next_token(s), attn_weights) -- predictions at
    EVERY position (including the last, which has no ground truth to check
    against but is still a valid "what comes after this" guess)."""
    tokens = np.asarray(tokens)
    single = tokens.ndim == 1
    if single:
        tokens = tokens[None, :]

    logits, weights, _ = forward(tokens, params)
    preds = logits.argmax(axis=-1)                        # (B,L)
    if single:
        return preds[0], weights[0]
    return preds, weights

## Run: train the model

Same hyperparameters as the script's `__main__` block: 900 steps, lr 0.5,
batch size 16, seed 0. Expect positions 1–6 to climb to ~100% accuracy
while position 0 stays near 0% (not ~17%, the 1/(6-1) chance level for
this vocabulary) — see "confidently wrong, not randomly wrong" above.

In [9]:
params = train(steps=900, lr=0.5, batch_size=16, vocab_size=VOCAB_SIZE,
                seq_len=SEQ_LEN, d_model=D_MODEL, seed=0, print_every=100)

print("\n--- final held-out accuracy (large sample) ---")
final_rng = np.random.default_rng(999)
final_tokens = make_batch(2000, VOCAB_SIZE, SEQ_LEN, rng=final_rng)
pos0_acc, pos1_6_acc, loss = position_accuracy(final_tokens, params)
print(f"loss {loss:.4f}")
print(f"position 0 accuracy (predicting token[1] from token[0] alone): {pos0_acc:.3f}"
      f"  -- b is unknowable from a single token (see \"why position 0 is unguessable\""
      f" above); in practice the shared identity-copy circuit"
      f" that solves positions 1-6 also fires on position 0's forced self-attention,"
      f" so it lands near 0% (confidently wrong) rather than ~{1 / (VOCAB_SIZE - 1):.2f} (randomly wrong) -- still not a bug")
print(f"positions 1-6 accuracy (both symbols already seen, pattern fully determined): {pos1_6_acc:.3f}"
      f"  -- expected ~1.000")

step  100   loss 0.6957   pos0 acc 0.000   pos1-6 acc 1.000
step  200   loss 0.6470   pos0 acc 0.000   pos1-6 acc 1.000


step  300   loss 0.6456   pos0 acc 0.000   pos1-6 acc 1.000


step  400   loss 0.6461   pos0 acc 0.000   pos1-6 acc 1.000
step  500   loss 0.6467   pos0 acc 0.000   pos1-6 acc 1.000


step  600   loss 0.6441   pos0 acc 0.000   pos1-6 acc 1.000


step  700   loss 0.6441   pos0 acc 0.000   pos1-6 acc 1.000
step  800   loss 0.6427   pos0 acc 0.000   pos1-6 acc 1.000


step  900   loss 0.6441   pos0 acc 0.000   pos1-6 acc 1.000

--- final held-out accuracy (large sample) ---


loss 0.6444
position 0 accuracy (predicting token[1] from token[0] alone): 0.000  -- b is unknowable from a single token (see "why position 0 is unguessable" above); in practice the shared identity-copy circuit that solves positions 1-6 also fires on position 0's forced self-attention, so it lands near 0% (confidently wrong) rather than ~0.20 (randomly wrong) -- still not a bug
positions 1-6 accuracy (both symbols already seen, pattern fully determined): 1.000  -- expected ~1.000


## Numerical gradient check

Finite differences vs. the analytic `backward()` above. This exact Q/K/V +
causal-mask math was already checked analytically-vs-numerically in a JS
prototype; this confirms the Python/NumPy port carries the same
correctness. In the standalone script this is gated behind `--gradcheck`
(so the default run trains instead); here it just runs directly.

In [10]:
def _gradient_check(epsilon=1e-5, n_checks=4, seed=0):
    rng = np.random.default_rng(seed)
    gc_params = init_params(seed=seed, d_model=D_MODEL, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN)
    tokens = make_batch(3, VOCAB_SIZE, SEQ_LEN, rng=rng)   # small batch is enough to exercise every path

    def loss_of(p):
        logits, _, cache = forward(tokens, p)
        n_pred = SEQ_LEN - 1
        return cross_entropy_loss(cache["probs"][:, :n_pred, :], tokens[:, 1:])

    _, _, cache = forward(tokens, gc_params)
    grads = backward(gc_params, cache, tokens)

    check_params = ["Wq", "Wk", "Wv", "Wout", "E_tok", "E_pos"]
    worst = 0.0
    for name in check_params:
        arr = gc_params[name]
        flat_idx = rng.choice(arr.size, size=min(n_checks, arr.size), replace=False)
        for fi in flat_idx:
            idx = np.unravel_index(fi, arr.shape)
            orig = arr[idx]
            arr[idx] = orig + epsilon
            loss_plus = loss_of(gc_params)
            arr[idx] = orig - epsilon
            loss_minus = loss_of(gc_params)
            arr[idx] = orig
            numeric = (loss_plus - loss_minus) / (2 * epsilon)
            analytic = grads[name][idx]
            rel_err = abs(numeric - analytic) / max(1e-8, abs(numeric) + abs(analytic))
            worst = max(worst, rel_err)
            status = "OK" if rel_err < 1e-4 else "MISMATCH"
            print(f"  {name}{idx}: numeric={numeric: .8f}  analytic={analytic: .8f}  rel_err={rel_err:.2e}  [{status}]")
    print(f"\nworst relative error across all checked entries: {worst:.2e}")
    assert worst < 1e-4, "gradient check failed"
    print("gradient check PASSED")


print("--- numerical gradient check (finite differences vs. backward()) ---")
_gradient_check()

--- numerical gradient check (finite differences vs. backward()) ---
  Wq(np.int64(3), np.int64(6)): numeric= 0.00006075  analytic= 0.00006075  rel_err=7.51e-08  [OK]
  Wq(np.int64(5), np.int64(6)): numeric=-0.00000464  analytic=-0.00000464  rel_err=4.26e-07  [OK]
  Wq(np.int64(4), np.int64(5)): numeric=-0.00000673  analytic=-0.00000673  rel_err=2.95e-07  [OK]
  Wq(np.int64(7), np.int64(5)): numeric=-0.00001624  analytic=-0.00001624  rel_err=1.42e-07  [OK]
  Wk(np.int64(5), np.int64(2)): numeric=-0.00002151  analytic=-0.00002151  rel_err=1.09e-07  [OK]
  Wk(np.int64(6), np.int64(3)): numeric= 0.00001805  analytic= 0.00001805  rel_err=8.05e-08  [OK]
  Wk(np.int64(2), np.int64(1)): numeric= 0.00003256  analytic= 0.00003256  rel_err=2.74e-07  [OK]
  Wk(np.int64(7), np.int64(1)): numeric=-0.00000086  analytic=-0.00000086  rel_err=1.43e-06  [OK]
  Wv(np.int64(0), np.int64(2)): numeric=-0.01467529  analytic=-0.01467529  rel_err=6.68e-10  [OK]
  Wv(np.int64(6), np.int64(0)): numeric=-0.002000

## Inference on held-out sequences

Same demo as the script's `__main__` block. Watch the attention weights at
the last position (`i=7`): they spread across **every** earlier position
carrying the "other" symbol, not just the nearest one — content-based
matching, not "N positions back". Position 2 is the cleanest illustration
of the mechanism (exactly one earlier position, `j=0`, carries the other
symbol, so there's no ambiguity about where the weight *should* go);
position 7 is the one that shows the actual multi-position spread.

In [11]:
print("--- sample predictions ---")
demo_rng = np.random.default_rng(123)
for i in range(3):
    seq = make_sequence(VOCAB_SIZE, SEQ_LEN, rng=demo_rng)
    preds, w = predict(seq, params)
    next_preds = preds[:-1]                 # prediction made at position i, for token[i+1]
    actual_next = seq[1:]
    correctness = " ".join("OK" if p == a else "no" for p, a in zip(next_preds, actual_next))
    print(f"sequence          : {list(seq)}")
    print(f"predicted next-tok: {list(next_preds)}  (predicted from tokens[0..i] at each position i)")
    print(f"actual next-tok   : {list(actual_next)}")
    print(f"correct?          : {correctness}   (position 0 is expected to say 'no' -- see above)")

    if i == 0:
        # Position 2 is the cleanest demo: exactly one earlier position
        # (j=0) carries the "other" symbol, and the causal mask forbids
        # looking past position 2 itself, so there's no ambiguity about
        # which position *should* get the weight -- one position back.
        i_query = 2
        print(f"\nattention weights at position {i_query} (query = token[{i_query}] = {seq[i_query]}):")
        for j in range(SEQ_LEN):
            bar = "#" * int(round(w[i_query, j] * 40))
            marker = "  <-- 1 back (correct source)" if j == i_query - 1 else ("  <-- self" if j == i_query else "")
            print(f"  attends to position {j} (token {seq[j]}): {w[i_query, j]:.3f} {bar}{marker}")

        # Later positions have *several* equally-valid earlier positions
        # carrying the same "other" symbol (period-2 series -> every
        # other position matches), so attention spreads across all of
        # them rather than collapsing onto exactly one -- still never
        # onto the same-symbol (even-offset) positions, which carry no
        # useful information for "what comes next".
        i_query = SEQ_LEN - 1
        print(f"\nattention weights at position {i_query} (query = token[{i_query}] = {seq[i_query]}) "
              f"-- note it spreads over EVERY earlier position carrying the other symbol, "
              f"not just the nearest one, because they are all equally informative:")
        for j in range(SEQ_LEN):
            bar = "#" * int(round(w[i_query, j] * 40))
            marker = "  <-- 1 back" if j == i_query - 1 else ("  <-- self" if j == i_query else "")
            print(f"  attends to position {j} (token {seq[j]}): {w[i_query, j]:.3f} {bar}{marker}")
    print()

--- sample predictions ---
sequence          : [np.int64(0), np.int64(4), np.int64(0), np.int64(4), np.int64(0), np.int64(4), np.int64(0), np.int64(4)]
predicted next-tok: [np.int64(0), np.int64(0), np.int64(4), np.int64(0), np.int64(4), np.int64(0), np.int64(4)]  (predicted from tokens[0..i] at each position i)
actual next-tok   : [np.int64(4), np.int64(0), np.int64(4), np.int64(0), np.int64(4), np.int64(0), np.int64(4)]
correct?          : no OK OK OK OK OK OK   (position 0 is expected to say 'no' -- see above)

attention weights at position 2 (query = token[2] = 0):
  attends to position 0 (token 0): 0.000 
  attends to position 1 (token 4): 1.000 ########################################  <-- 1 back (correct source)
  attends to position 2 (token 0): 0.000   <-- self
  attends to position 3 (token 4): 0.000 
  attends to position 4 (token 0): 0.000 
  attends to position 5 (token 4): 0.000 
  attends to position 6 (token 0): 0.000 
  attends to position 7 (token 4): 0.000 

attentio

## Optional: attention heatmap

The full `(L,L)` causal attention matrix for one example sequence, if
`matplotlib` is available (this cell is skipped otherwise, rather than
adding a new dependency to the course). The upper triangle should be
visibly zero (masked); the lower triangle should show the content-matching
pattern from the section above — weight concentrated on same-symbol
earlier positions, spread across several of them for later queries.

In [12]:
try:
    import matplotlib
    HAVE_MPL = True
except ImportError:
    HAVE_MPL = False

if HAVE_MPL:
    import matplotlib.pyplot as plt

    heat_rng = np.random.default_rng(7)
    heat_seq = make_sequence(VOCAB_SIZE, SEQ_LEN, rng=heat_rng)
    _, hw = predict(heat_seq, params)

    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(hw, cmap="viridis", vmin=0, vmax=1)
    ax.set_xlabel("key position j")
    ax.set_ylabel("query position i")
    ax.set_title(f"causal attention weights, sequence {list(heat_seq)}")
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    fig.colorbar(im, ax=ax, label="attention weight")
    plt.show()
else:
    print("matplotlib not available -- skipping heatmap (no new dependency added).")

matplotlib not available -- skipping heatmap (no new dependency added).
